# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all structures by their `@id` fields per the Croissant standard.

### Dataset Source
The dataset is described by a Croissant schema accessible at this URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not present
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. Dataset entities will always be referenced by their `@id` fields

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")print(f"License: {metadata.license}")

## 2. Data Overview

List all available record sets in the dataset by their `@id` and `name`, and list available fields for each. This helps us choose which record set and fields to extract and analyze next.

In [ ]:
# List all record sets available in the dataset (by @id)

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    print(f"Record Sets found: {len(metadata.recordSet)}\n")
    for record_set in metadata.recordSet:
        print(f"Record Set @id: {record_set['@id']}, name: {record_set.get('name','<no name>')}")
        if 'field' in record_set:
            print("  Fields:")
            for field in record_set['field']:
                # Each field is a dict with '@id', 'name', 'dataType', etc.
                print(f"    @id: {field['@id']}\tname: {field.get('name','')}\ttype: {field.get('dataType','')}")
        print("")
else:
    print("No record sets found in Croissant metadata.")

## 3. Data Extraction

Extract data from a specific record set, referencing it and its fields by their `@id`. For small datasets, it is convenient to load the whole record set into a `pandas` DataFrame for exploration.

First, we'll list all record set `@id`s for reference. Then, select the main tabular record set (usually the first or largest) for demonstration.

In [ ]:
# Retrieve all record set @ids
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = [r['@id'] for r in metadata.recordSet]
else:
    print("No record sets detected!")
    record_sets = []

# Use the first record set for further demonstration
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Main record set @id for extraction: {main_record_set_id}")
else:
    main_record_set_id = None

dataframes = dict()

for rs_id in record_sets:
    print(f"Loading records for record set {rs_id}")
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Example rows:\n{df.head(2)}\n")

if main_record_set_id is not None:
    print(f"Available columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

For demonstration, we'll pick common numeric and categorical fields by their `@id`, perform some filtering, normalization, and group by analyses.

**Remember**: All columns and fields must be referenced by their `@id`.

In [ ]:
# Example EDA: filtering and normalizing a numeric column

# First, select a numeric field from the main record set by @id
main_fields = None
main_numeric_field_id = None
group_field_id = None

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # Get the full structure for main record set
    for rset in metadata.recordSet:
        if rset['@id'] == main_record_set_id:
            main_fields = rset.get('field', [])
            break

    # Choose a numeric field (e.g., age or interval in months, etc.) by dataType if available
    for field in main_fields:
        if 'dataType' in field and 'Integer' in field['dataType'] or 'Float' in field['dataType']:
            main_numeric_field_id = field['@id']
            break
    # Choose a categorical field for grouping (e.g., 'Sex', 'CancerType', etc.)
    for field in main_fields:
        if 'dataType' in field and field['dataType'] == 'Text' and field['@id'] != main_numeric_field_id:
            group_field_id = field['@id']
            break

print(f"Selected numeric field for analysis: {main_numeric_field_id}")
print(f"Selected group field: {group_field_id}")

df = dataframes[main_record_set_id]

# Only proceed if selected fields are found
if main_numeric_field_id in df.columns:
    # Only consider rows where numeric field is not missing
    filtered_df = df[df[main_numeric_field_id].notna()].copy()

    # Filter: show records where value > threshold (e.g., threshold=50 for age)
    threshold = filtered_df[main_numeric_field_id].mean()
    filtered_df = filtered_df[filtered_df[main_numeric_field_id] > threshold].copy()

    print(f"\nFiltered records with {main_numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[main_numeric_field_id]].head())

    # Normalize
    norm_col = f"{main_numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[main_numeric_field_id] - 
                             filtered_df[main_numeric_field_id].mean()) / filtered_df[main_numeric_field_id].std()
    print(f"\nNormalized {main_numeric_field_id} for filtered records:")
    print(filtered_df[[main_numeric_field_id, norm_col]].head())

    # Group by a categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[main_numeric_field_id].mean()
        print(f"\nGrouped mean {main_numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field selected or field missing from DataFrame.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and breakdown by a key category if group field exists.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of numeric field
if main_numeric_field_id is not None and main_numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[main_numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {main_numeric_field_id}")
    plt.xlabel(main_numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
    
    # Boxplot grouped by categorical
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=main_numeric_field_id, data=df)
        plt.title(f"{main_numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² dataset using the `mlcroissant` library, following strict referencing of all data entities by their `@id`s.
- We reviewed all record sets and their available fields by `@id`, then extracted the main data into a DataFrame.
- Standard EDA steps such as filtering, normalization, grouping, and visualizing were performed using only Croissant-compliant column keys.

**This notebook can be adapted for other Croissant datasets by adjusting the schema URL and field selections, maintaining strict compliance with referencing by `@id`.**